# 03 — Volatility Forecasting

This notebook compares simple and parametric volatility forecasts and produces the monthly volatility input used by the portfolio backtest.

## 0. Setup and helper functions


In [39]:
from pathlib import Path

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_arch
from arch import arch_model


In [40]:
default_layout = dict(
    template="plotly_white",
    height=500,
    margin=dict(t=70, l=60, r=40, b=60),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
)

In [41]:
returns = pd.read_csv("data/returns.csv", index_col=0, parse_dates=True)

IS_END = "2017-12-31"
OOS_START = "2018-01-01"
HORIZON = 21
TRADING_DAYS = 252
CASH_PROXY = "SHY"
risk_model_assets = [asset for asset in returns.columns if asset != CASH_PROXY]

OUTPUT_DIR = Path("results/volatility_forecast")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [42]:
def evaluate_vol_forecast(vol_forecast, realized_vol, eps=1e-8):
    """Compare volatility forecasts with the forward realized-volatility target."""
    common_index = vol_forecast.index.intersection(realized_vol.index)
    common_cols = vol_forecast.columns.intersection(realized_vol.columns)

    forecast = vol_forecast.loc[common_index, common_cols]
    realized = realized_vol.loc[common_index, common_cols]

    error = forecast - realized

    forecast_var = forecast.pow(2).clip(lower=eps)
    realized_var = realized.pow(2).clip(lower=eps)

    ratio = realized_var / forecast_var

    qlike = (
        ratio
        - np.log(ratio)
        - 1
    ).mean()

    summary = pd.DataFrame({
        "MAE": error.abs().mean(),
        "RMSE": np.sqrt((error ** 2).mean()),
        "Bias": error.mean(),
        "Correlation": forecast.corrwith(realized),
        "QLIKE": qlike
    })

    return summary.round(4)


In [43]:
def vol_forecast_regression_hac(vol_forecast, realized_vol, maxlags=21):
    """
    Run asset-level calibration regressions with HAC standard errors.

        realized_vol_t = alpha + beta * forecast_vol_t + error_t

    HAC errors are used because the 21-day realized-volatility target
    overlaps across adjacent forecast dates.
    """

    common_index = vol_forecast.index.intersection(realized_vol.index)
    common_cols = vol_forecast.columns.intersection(realized_vol.columns)

    results = []

    for asset in common_cols:
        data = pd.DataFrame({
            "realized": realized_vol.loc[common_index, asset],
            "forecast": vol_forecast.loc[common_index, asset]
        }).dropna()

        y = data["realized"]
        X = sm.add_constant(data["forecast"])

        model = sm.OLS(y, X).fit(
            cov_type="HAC",
            cov_kwds={"maxlags": maxlags}
        )

        # Joint calibration check: zero intercept and unit forecast loading.
        joint_test = model.f_test("const = 0, forecast = 1")

        results.append({
            "Asset": asset,
            "Alpha": model.params["const"],
            "Beta": model.params["forecast"],
            "Alpha p-value": model.pvalues["const"],
            "Beta p-value": model.pvalues["forecast"],
            "R2": model.rsquared,
            "Joint F-stat": float(joint_test.fvalue),
            "Joint p-value": float(joint_test.pvalue),
        })

    return pd.DataFrame(results).set_index("Asset").round(4)


In [44]:
def run_arch_lm_tests(returns_data, lags=5):
    """Run asset-level ARCH-LM tests on demeaned returns."""

    results = []

    for asset in returns_data.columns:
        r = returns_data[asset].dropna()

        # Demean returns before testing for ARCH effects.
        residuals = r - r.mean()

        lm_stat, lm_pvalue, f_stat, f_pvalue = het_arch(
            residuals,
            nlags=lags
        )

        results.append({
            "Asset": asset,
            "ARCH-LM Stat": lm_stat,
            "ARCH-LM p-value": lm_pvalue,
            "F-stat": f_stat,
            "F-test p-value": f_pvalue,
            "Lags": lags,
            "N obs": len(r)
        })

    return pd.DataFrame(results).set_index("Asset").round(4)


In [45]:
def fit_garch(returns_data, vol="GARCH", p=1, o=0, q=1, dist="normal"):
    """Fit GARCH-family models and return fitted models plus diagnostics."""

    models, rows = {}, []

    for asset in returns_data:
        model = arch_model(
            returns_data[asset].dropna(),
            mean="Constant",
            vol=vol,
            p=p,
            o=o,
            q=q,
            dist=dist,
            rescale=False,
        ).fit(disp="off")

        models[asset] = model

        params = model.params
        alpha = params.get("alpha[1]", np.nan)
        beta = params.get("beta[1]", np.nan)
        gamma = params.get("gamma[1]", np.nan)
        conv_flag = getattr(model, "convergence_flag", np.nan)

        row = {
            "Asset": asset,
            "Omega": params.get("omega", np.nan),
            "Alpha": alpha,
            "Beta": beta,
            "Nu": params.get("nu", np.nan),
            "Converged": bool(conv_flag == 0) if pd.notna(conv_flag) else np.nan,
            "LogLik": model.loglikelihood,
            "AIC": model.aic,
            "BIC": model.bic,
        }

        if o:
            row.update({
                "Gamma": gamma,
                "Persistence": alpha + 0.5 * gamma + beta,
                "Gamma p-value": model.pvalues.get("gamma[1]", np.nan),
            })
        else:
            row["Alpha + Beta"] = alpha + beta

        rows.append(row)

    return models, pd.DataFrame(rows).set_index("Asset").round(4)

In [46]:
def forecast_sanity_table(vol_forecast):
    """Summarize the distribution of forecast volatilities by asset."""
    return pd.DataFrame({
        "Min": vol_forecast.min(),
        "Median": vol_forecast.median(),
        "Mean": vol_forecast.mean(),
        "Max": vol_forecast.max()
    }).round(4)


In [47]:
def garch_oos_forecast(
    returns_data,
    model_type="GARCH",
    p=1,
    o=0,
    q=1,
    dist="t",
    is_end="2017-12-31",
    oos_start="2018-01-01",
    horizon=21
):
    """Generate OOS annualized volatility forecasts from fixed GARCH parameters."""
    forecasts = {}
    fitted_models = {}

    for asset in returns_data.columns:
        r = returns_data[asset].dropna()

        model = arch_model(
            r,
            mean="Constant",
            vol=model_type,
            p=p,
            o=o,
            q=q,
            dist=dist,
            rescale=False
        )

        fitted = model.fit(last_obs=is_end, disp="off")
        fitted_models[asset] = fitted

        forecast = fitted.forecast(
            start=oos_start,
            horizon=horizon,
            reindex=True,
            align="origin"
        )

        # Average expected daily variance across the forecast horizon.
        avg_var_forecast = forecast.variance.mean(axis=1)

        # Convert percent daily variance to annualized decimal volatility.
        forecasts[asset] = np.sqrt(avg_var_forecast) / 100 * np.sqrt(TRADING_DAYS)

    forecasts = pd.DataFrame(forecasts).loc[oos_start:]

    return forecasts.dropna(how="all"), fitted_models


## 1. Realized Volatility Target and Rolling Baseline

Forecasts are evaluated against a forward 21-trading-day realized-volatility target. For asset $i$ and forecast date $t$, the target uses returns from $t+1$ to $t+21$:

$$
RV_{i,t:t+21} = \sum_{j=1}^{21} r_{i,t+j}^{2}.
$$

The realized variance is converted to annualized realized volatility as:

$$
\sigma^{\mathrm{realized}}_{i,t:t+21}
=
\sqrt{\frac{252}{21} RV_{i,t:t+21}}.
$$

This target is ex-post and is used only for evaluation. Forecast inputs remain dated at $t$ or earlier; the target measures what is subsequently realized. Keeping these two objects separate avoids look-ahead in model construction and makes the timing of the evaluation explicit.


In [48]:
realized_var_21d = (
    returns.pow(2)
    .rolling(window=HORIZON)
    .sum()
    .shift(-HORIZON)
)

realized_vol_21d = np.sqrt(realized_var_21d * TRADING_DAYS / HORIZON)

is_index = returns.loc[:IS_END].index
IS_EVAL_END = is_index[-HORIZON - 1]

realized_vol_is = realized_vol_21d.loc[:IS_EVAL_END]
realized_vol_oos = realized_vol_21d.loc[OOS_START:].dropna(how="all")


### Rolling Historical Volatility Baseline

The first benchmark is a 63-day rolling standard deviation of daily returns, annualized with 252 trading days. It is included as a transparent baseline for the model-selection exercise.

The estimate is shifted by one day, so the forecast at date $t$ uses only returns observed up to $t-1$. The same daily annualized volatility estimate is evaluated against the forward 21-day realized target, which treats the current risk estimate as the expected average risk level over the next month.

This benchmark is simple and stable, but it has two predictable drawbacks: it weights every observation in the window equally, and it reacts only after large moves enter the rolling window.


In [49]:
ROLLING_WINDOW = 63

In [50]:
rolling_vol = returns.rolling(window=ROLLING_WINDOW).std() * np.sqrt(TRADING_DAYS)
rolling_vol_lagged = rolling_vol.shift(1)
rolling_vol_lagged.tail()

,SPY,EFA,EEM,IEF,TLT,LQD,HYG,GLD,DBC,VNQ,SHY
Date,,,,,,,,,,,
2026-06-03,0.141872,0.212134,0.292856,0.054402,0.100660,0.068031,0.056371,0.261981,0.247452,0.148296,0.015695
2026-06-04,0.142557,0.211525,0.294097,0.054474,0.100764,0.068208,0.056273,0.261708,0.247199,0.148296,0.015679
2026-06-05,0.141867,0.207053,0.290527,0.054264,0.100647,0.067907,0.055763,0.261762,0.247999,0.151089,0.015679
2026-06-08,0.149354,0.212925,0.320712,0.055203,0.100879,0.068723,0.055815,0.268261,0.241880,0.149886,0.016193
2026-06-09,0.148643,0.212793,0.320339,0.054782,0.099588,0.067621,0.054604,0.268466,0.240467,0.152586,0.016194


In [51]:
asset = "SPY"

vol_compare = pd.DataFrame({
    "Rolling 63D Forecast": rolling_vol_lagged[asset].loc[:IS_EVAL_END],
    "Forward 21D Realized Vol": realized_vol_is[asset]
}).dropna()

In [52]:
fig = go.Figure()

for col in vol_compare.columns:
    fig.add_trace(
        go.Scatter(
            x=vol_compare.index,
            y=vol_compare[col],
            mode="lines",
            name=col
        )
    )

fig.update_layout(
    title=f"{asset}: Rolling Volatility Forecast vs Forward Realized Volatility",
    xaxis_title="Date",
    yaxis_title="Annualized Volatility",
    **default_layout
)

fig.show()

The SPY chart is shown for the in-sample period to keep the visual check separate from OOS evaluation. The rolling forecast captures the main volatility regimes, including 2008-2009 and the sovereign-credit stress period, but it reacts with a delay and remains elevated after shocks fade. That lag is the main weakness of a fixed-window estimator.


### Error Metrics

Forecasts are evaluated with MAE, RMSE, bias, forecast-realized correlation, and QLIKE. MAE and RMSE summarize error in volatility units, bias shows average over- or under-forecasting, correlation measures timing, and QLIKE evaluates forecasts in variance space. Lower MAE, RMSE, absolute bias, and QLIKE are preferred; higher correlation is preferred.


In [53]:
evaluate_vol_forecast(rolling_vol_lagged.loc[:IS_EVAL_END], realized_vol_is)

,MAE,RMSE,Bias,Correlation,QLIKE
SPY,0.0570,0.0964,0.0071,0.6563,0.3873
EFA,0.0661,0.1149,0.0082,0.6392,0.3705
EEM,0.0853,0.1601,0.0100,0.6494,0.3217
IEF,0.0142,0.0188,0.0013,0.6587,0.1401
TLT,0.0293,0.0415,0.0023,0.6020,0.1554
LQD,0.0218,0.0593,0.0029,0.4199,0.6155
HYG,0.0385,0.0712,0.0051,0.6421,0.8270
GLD,0.0440,0.0652,0.0042,0.6287,0.2620
DBC,0.0414,0.0555,0.0026,0.7730,0.1862
VNQ,0.0822,0.1425,0.0097,0.8181,0.3141


The rolling benchmark carries useful signal: correlations are positive for every asset. Errors are lowest for short-duration fixed income and highest for assets exposed to larger regime shifts, such as EEM, VNQ, and SPY.

The positive bias indicates mild over-forecasting on average. For allocation, that is less concerning than persistent under-forecasting, but the delayed response around stress periods leaves room for a more adaptive benchmark.


## 2. EWMA / RiskMetrics Volatility Forecast

EWMA is added as a second benchmark. It keeps the historical-volatility logic but gives more weight to recent squared returns, allowing forecasts to update faster after market moves.

I use $\lambda = 0.94$, the standard daily RiskMetrics setting. The forecast is shifted by one day, so the date-$t$ estimate uses information available before $t$.


In [54]:
LAMBDA_EWMA = 0.94

ewma_var = returns.pow(2).ewm(alpha=1 - LAMBDA_EWMA, adjust=False).mean()
ewma_vol = np.sqrt(ewma_var) * np.sqrt(TRADING_DAYS)

ewma_vol_lagged = ewma_vol.shift(1)

In [55]:
asset = "SPY"

vol_compare_spy = pd.DataFrame({
    "Rolling 63D Forecast": rolling_vol_lagged[asset].loc[:IS_EVAL_END],
    "Forward 21D Realized Vol": realized_vol_is[asset],
    "EWMA Forecast": ewma_vol_lagged[asset].loc[:IS_EVAL_END]
}).dropna()

fig = go.Figure()

for col in vol_compare_spy.columns:
    fig.add_trace(
        go.Scatter(
            x=vol_compare_spy.index,
            y=vol_compare_spy[col],
            mode="lines",
            name=col
        )
    )

fig.update_layout(
    title=f"{asset}: Volatility Forecasts vs Forward Realized Volatility",
    xaxis_title="Date",
    yaxis_title="Annualized Volatility",
    **default_layout
)

fig.show()

In [56]:
evaluate_vol_forecast(ewma_vol_lagged.loc[:IS_EVAL_END], realized_vol_is)

,MAE,RMSE,Bias,Correlation,QLIKE
SPY,0.0512,0.0831,0.0033,0.7452,0.3633
EFA,0.0588,0.0991,0.0041,0.7331,0.3261
EEM,0.0739,0.1306,0.0053,0.7652,0.2784
IEF,0.0140,0.0189,0.0004,0.6722,0.1535
TLT,0.0286,0.0405,0.0005,0.6470,0.3005
LQD,0.0204,0.0550,0.0015,0.4995,0.6437
HYG,0.0351,0.0659,0.0024,0.6951,0.9438
GLD,0.0414,0.0609,0.0019,0.6841,0.2643
DBC,0.0398,0.0530,0.0016,0.7970,0.1639
VNQ,0.0721,0.1198,0.0042,0.8692,0.2889


For SPY, EWMA responds faster than the 63-day rolling estimate around 2008-2009. It still does not fully match the timing or size of realized volatility spikes, but it reduces the fixed-window lag.

The cross-asset table shows the same trade-off. EWMA improves responsiveness while remaining a benchmark rather than a final model: errors are still material for higher-volatility assets, and calibration has to be checked directly.


### Forecast Validation Regression

I use asset-level calibration regressions to check whether the forecast is informative and whether its scale is aligned with realized volatility:

$$
\sigma^{\mathrm{realized}}_{i,t:t+21}
=
\alpha_i
+
\beta_i \hat{\sigma}_{i,t}
+u_{i,t}.
$$

A positive $\beta_i$ indicates timing information. The joint test $\alpha_i = 0$ and $\beta_i = 1$ is a stricter calibration check. HAC/Newey-West standard errors are used because the 21-day realized target overlaps across adjacent dates.


In [57]:
ewma_regression = vol_forecast_regression_hac(
    vol_forecast=ewma_vol_lagged.loc[:IS_EVAL_END],
    realized_vol=realized_vol_is,
    maxlags=HORIZON
)

ewma_regression

,Alpha,Beta,Alpha p-value,Beta p-value,R2,Joint F-stat,Joint p-value
Asset,,,,,,,
SPY,0.0341,0.7758,0.0147,0.0000,0.5553,3.0135,0.0493
EFA,0.0439,0.7653,0.0118,0.0000,0.5374,3.2059,0.0407
EEM,0.0500,0.7914,0.0910,0.0000,0.5856,1.5096,0.2212
IEF,0.0186,0.7143,0.0000,0.0000,0.4519,12.3810,0.0000
TLT,0.0447,0.6807,0.0000,0.0000,0.4186,20.6138,0.0000
LQD,0.0307,0.5164,0.0000,0.0002,0.2495,8.7193,0.0002
HYG,0.0237,0.7174,0.0028,0.0000,0.4831,4.5636,0.0105
GLD,0.0467,0.7240,0.0000,0.0000,0.4680,8.9182,0.0001
DBC,0.0290,0.8317,0.0025,0.0000,0.6352,4.5596,0.0105


EWMA is informative in-sample: beta is positive and statistically significant for every asset. Explanatory power is strongest for VNQ, SHY, DBC, EEM, and EFA, and weaker for LQD and TLT.

The joint calibration test is rejected for most assets, so EWMA should not be treated as unbiased in level. It is a useful adaptive benchmark, but the fixed decay parameter leaves no room for asset-specific shock sensitivity or persistence.


## 3. GARCH Model Selection

The next step is to estimate asset-level GARCH specifications. The goal is not to overfit each ETF, but to test whether a common, interpretable volatility model is justified by the in-sample return dynamics.

Before fitting GARCH, I check for ARCH effects in returns. This is a diagnostic step: if returns show volatility clustering, conditional-volatility models are more defensible than a fixed-window estimate alone.


### ARCH-LM Test

The ARCH-LM test is used as a pre-check for volatility clustering. For each asset, demeaned returns are used to test whether lagged squared residuals explain current squared residuals:

$$
\hat{\varepsilon}_{i,t}^{2}
=
\alpha_0
+
\sum_{k=1}^{q} \alpha_k \hat{\varepsilon}_{i,t-k}^{2}
+u_{i,t}.
$$

The null is no ARCH effects. Rejection supports conditional-volatility modelling for that asset.


In [58]:
returns_pct = returns.mul(100).dropna(how="all")
returns_pct_is = returns_pct.loc[:IS_END]
returns_pct_oos = returns_pct.loc[OOS_START:]

In [59]:
arch_lm_results_5 = run_arch_lm_tests(
    returns_data=returns_pct_is,
    lags=5
)

arch_lm_results_5

,ARCH-LM Stat,ARCH-LM p-value,F-stat,F-test p-value,Lags,N obs
Asset,,,,,,
SPY,639.7641,0.0,167.3899,0.0,5,2701
EFA,547.9050,0.0,137.2253,0.0,5,2701
EEM,715.4184,0.0,194.3344,0.0,5,2701
IEF,116.1881,0.0,24.2301,0.0,5,2701
TLT,238.0607,0.0,52.1073,0.0,5,2701
LQD,1011.4962,0.0,323.0536,0.0,5,2701
HYG,341.3638,0.0,77.9966,0.0,5,2701
GLD,103.9196,0.0,21.5691,0.0,5,2701
DBC,404.4447,0.0,94.9535,0.0,5,2701


The ARCH-LM results reject the no-ARCH null across the universe. This supports including conditional-volatility models in the comparison rather than relying only on rolling historical volatility.


In [60]:
arch_lm_results_10 = run_arch_lm_tests(
    returns_data=returns_pct_is,
    lags=10
)

arch_lm_results_10

,ARCH-LM Stat,ARCH-LM p-value,F-stat,F-test p-value,Lags,N obs
Asset,,,,,,
SPY,737.8565,0.0,101.2448,0.0,10,2701
EFA,704.0807,0.0,94.9679,0.0,10,2701
EEM,1016.9368,0.0,162.8009,0.0,10,2701
IEF,155.6448,0.0,16.4525,0.0,10,2701
TLT,276.4669,0.0,30.6863,0.0,10,2701
LQD,1454.2281,0.0,315.1213,0.0,10,2701
HYG,692.7461,0.0,92.9091,0.0,10,2701
GLD,170.4416,0.0,18.1223,0.0,10,2701
DBC,490.4556,0.0,59.7316,0.0,10,2701


### Baseline Gaussian GARCH(1,1)

The baseline specification is Gaussian GARCH(1,1). It provides the reference point for the Student-t and GJR extensions:

$$
r_{i,t} = \mu_i + \varepsilon_{i,t},
\qquad
\varepsilon_{i,t} = \sigma_{i,t} z_{i,t}
$$

$$
\sigma_{i,t}^{2}
=
\omega_i
+
\alpha_i \varepsilon_{i,t-1}^{2}
+
\beta_i \sigma_{i,t-1}^{2}.
$$

For model selection, the key diagnostics are shock sensitivity ($\alpha$), persistence ($\beta$), total persistence ($\alpha + \beta$), likelihood-based criteria, and convergence behavior.


In [61]:
garch_norm_models, garch_norm_results = fit_garch(returns_pct_is)

garch_norm_results

,Omega,Alpha,Beta,Nu,Converged,LogLik,AIC,BIC,Alpha + Beta
Asset,,,,,,,,,
SPY,0.0202,0.1287,0.8578,NaN,True,-3634.6285,7277.2569,7300.8624,0.9865
EFA,0.0119,0.1126,0.8874,NaN,True,-4201.7400,8411.4800,8435.0855,1.0000
EEM,0.0247,0.0976,0.8967,NaN,True,-4887.2567,9782.5135,9806.1190,0.9944
IEF,0.0013,0.0411,0.9525,NaN,True,-1427.8147,2863.6294,2887.2349,0.9936
TLT,0.0109,0.0466,0.9408,NaN,True,-3486.1644,6980.3287,7003.9342,0.9874
LQD,0.0063,0.1391,0.8364,NaN,True,-1280.3843,2568.7687,2592.3742,0.9755
HYG,0.0042,0.1543,0.8457,NaN,True,-1861.3554,3730.7109,3754.3164,1.0000
GLD,0.0126,0.0502,0.9414,NaN,True,-4043.8493,8095.6985,8119.3040,0.9915
DBC,0.0041,0.0427,0.9552,NaN,True,-4062.7747,8133.5495,8157.1550,0.9978


The Gaussian GARCH estimates show high persistence across the universe, with $alpha + beta$ close to one for most assets. That confirms that volatility shocks decay slowly and that a conditional-volatility model is appropriate.

The baseline is useful, but not final. Several assets sit very close to the persistence boundary, and Gaussian innovations may understate tail risk. That motivates testing Student-t innovations next.


### Student-t GARCH(1,1)

Student-t innovations are tested to allow fatter residual tails while keeping the same GARCH(1,1) variance recursion:

$$
\varepsilon_{i,t} = \sigma_{i,t} z_{i,t},
\qquad
z_{i,t} \sim t_{\nu_i}.
$$

This is a specification check: if information criteria improve without creating unstable estimates, the distributional assumption is better aligned with the return data.


In [62]:
garch_t_models, garch_t_results = fit_garch(returns_pct_is, vol="GARCH", p=1, o=0, q=1, dist="t")

garch_t_results

,Omega,Alpha,Beta,Nu,Converged,LogLik,AIC,BIC,Alpha + Beta
Asset,,,,,,,,,
SPY,0.0132,0.1330,0.8670,5.3218,True,-3564.4202,7138.8405,7168.3474,1.0000
EFA,0.0105,0.1038,0.8962,6.8995,True,-4151.7140,8313.4280,8342.9349,1.0000
EEM,0.0240,0.0902,0.9036,9.8861,True,-4864.8403,9739.6806,9769.1874,0.9938
IEF,0.0010,0.0418,0.9532,13.8971,True,-1410.0976,2830.1953,2859.7022,0.9950
TLT,0.0078,0.0437,0.9475,15.2180,True,-3472.5923,6955.1846,6984.6915,0.9912
LQD,0.0026,0.0589,0.9245,7.0930,True,-1165.0713,2340.1427,2369.6496,0.9835
HYG,0.0035,0.1524,0.8476,4.5708,True,-1691.1219,3392.2439,3421.7508,1.0000
GLD,0.0073,0.0411,0.9544,5.3872,True,-3945.1944,7900.3887,7929.8956,0.9955
DBC,0.0031,0.0461,0.9529,9.1667,True,-4029.3373,8068.6745,8098.1814,0.9991


The Student-t model raises a convergence warning for SHY, and its persistence is slightly above one. The information-criterion values for SHY are therefore not interpretable.

This is not a portfolio-wide failure. SHY is a low-volatility cash proxy, and the additional tail parameter is poorly identified for this series. I keep the Student-t comparison for the risk assets and treat SHY separately in the final forecast construction.


In [63]:
garch_dist_comparison = pd.DataFrame({
    "Gaussian AIC": garch_norm_results["AIC"],
    "Student-t AIC": garch_t_results["AIC"],
    "AIC Improvement": garch_norm_results["AIC"] - garch_t_results["AIC"],
    "Gaussian BIC": garch_norm_results["BIC"],
    "Student-t BIC": garch_t_results["BIC"],
    "BIC Improvement": garch_norm_results["BIC"] - garch_t_results["BIC"],
}).round(4)

garch_dist_comparison.loc[risk_model_assets]

,Gaussian AIC,Student-t AIC,AIC Improvement,Gaussian BIC,Student-t BIC,BIC Improvement
Asset,,,,,,
SPY,7277.2569,7138.8405,138.4164,7300.8624,7168.3474,132.5150
EFA,8411.4800,8313.4280,98.0520,8435.0855,8342.9349,92.1506
EEM,9782.5135,9739.6806,42.8329,9806.1190,9769.1874,36.9316
IEF,2863.6294,2830.1953,33.4341,2887.2349,2859.7022,27.5327
TLT,6980.3287,6955.1846,25.1441,7003.9342,6984.6915,19.2427
LQD,2568.7687,2340.1427,228.6260,2592.3742,2369.6496,222.7246
HYG,3730.7109,3392.2439,338.4670,3754.3164,3421.7508,332.5656
GLD,8095.6985,7900.3887,195.3098,8119.3040,7929.8956,189.4084
DBC,8133.5495,8068.6745,64.8750,8157.1550,8098.1814,58.9736


Excluding SHY, the information-criterion comparison favors Student-t innovations for every asset. The improvement is strongest for HYG, GLD, LQD, SPY, and EFA, which supports using a fat-tailed innovation distribution for the risk assets.

The conclusion should be stated carefully: Student-t improves in-sample likelihood fit, but it still has to prove useful in the out-of-sample forecast comparison.


### Testing Asymmetric Volatility Responses

The final candidate is GJR-GARCH with Student-t innovations. It tests whether negative shocks have a different impact on next-period volatility than positive shocks of the same size, which is most relevant for equity and credit assets:

$$
\sigma_{i,t}^{2}
=
\omega_i
+
\alpha_i \varepsilon_{i,t-1}^{2}
+
\gamma_i \varepsilon_{i,t-1}^{2} I(\varepsilon_{i,t-1}<0)
+
\beta_i \sigma_{i,t-1}^{2}.
$$

A positive $\gamma_i$ means negative shocks raise conditional volatility more than positive shocks of the same magnitude. The question is whether that extra parameter improves the model enough to justify using it in portfolio inputs.


In [64]:
gjr_t_models, gjr_t_results = fit_garch(returns_pct_is, vol="GARCH", p=1, o=1, q=1, dist="t")

gjr_t_results

,Omega,Alpha,Beta,Nu,Converged,LogLik,AIC,BIC,Gamma,Persistence,Gamma p-value
Asset,,,,,,,,,,,
SPY,0.0172,0.0000,0.8647,5.7324,True,-3510.5907,7033.1814,7068.5896,0.2565,0.9929,0.0000
EFA,0.0112,0.0126,0.9085,7.0914,True,-4123.4167,8258.8335,8294.2418,0.1487,0.9954,0.0000
EEM,0.0182,0.0105,0.9253,10.9843,True,-4841.0505,9694.1010,9729.5093,0.1162,0.9940,0.0000
IEF,0.0010,0.0450,0.9537,13.8452,True,-1409.8465,2831.6930,2867.1013,-0.0075,0.9949,0.4661
TLT,0.0068,0.0537,0.9529,15.5682,True,-3469.6291,6951.2583,6986.6666,-0.0273,0.9929,0.0117
LQD,0.0025,0.0378,0.9302,7.1810,True,-1163.3138,2338.6275,2374.0358,0.0292,0.9826,0.0578
HYG,0.0025,0.0367,0.8728,4.7767,True,-1655.6903,3323.3805,3358.7888,0.1810,1.0000,0.0000
GLD,0.0074,0.0503,0.9549,5.3749,True,-3943.7916,7899.5832,7934.9915,-0.0185,0.9959,0.0901
DBC,0.0025,0.0301,0.9555,9.2534,True,-4025.8931,8063.7862,8099.1944,0.0272,0.9992,0.0118


SHY again produces unstable GARCH estimates. I exclude SHY from the economic interpretation of the GJR comparison and keep it assigned to EWMA in the final construction.


In [65]:
asymmetry_comparison = pd.DataFrame({
    "GARCH-t AIC": garch_t_results["AIC"],
    "GJR-t AIC": gjr_t_results["AIC"],
    "AIC Improvement": garch_t_results["AIC"] - gjr_t_results["AIC"],
    "GARCH-t BIC": garch_t_results["BIC"],
    "GJR-t BIC": gjr_t_results["BIC"],
    "BIC Improvement": garch_t_results["BIC"] - gjr_t_results["BIC"],
    "Gamma": gjr_t_results["Gamma"],
    "Gamma p-value": gjr_t_results["Gamma p-value"]
}).round(4)

asymmetry_comparison.loc[risk_model_assets]

,GARCH-t AIC,GJR-t AIC,AIC Improvement,GARCH-t BIC,GJR-t BIC,BIC Improvement,Gamma,Gamma p-value
Asset,,,,,,,,
SPY,7138.8405,7033.1814,105.6591,7168.3474,7068.5896,99.7578,0.2565,0.0000
EFA,8313.4280,8258.8335,54.5945,8342.9349,8294.2418,48.6931,0.1487,0.0000
EEM,9739.6806,9694.1010,45.5796,9769.1874,9729.5093,39.6781,0.1162,0.0000
IEF,2830.1953,2831.6930,-1.4977,2859.7022,2867.1013,-7.3991,-0.0075,0.4661
TLT,6955.1846,6951.2583,3.9263,6984.6915,6986.6666,-1.9751,-0.0273,0.0117
LQD,2340.1427,2338.6275,1.5152,2369.6496,2374.0358,-4.3862,0.0292,0.0578
HYG,3392.2439,3323.3805,68.8634,3421.7508,3358.7888,62.9620,0.1810,0.0000
GLD,7900.3887,7899.5832,0.8055,7929.8956,7934.9915,-5.0959,-0.0185,0.0901
DBC,8068.6745,8063.7862,4.8883,8098.1814,8099.1944,-1.0130,0.0272,0.0118


The GJR results support asymmetry for the equity-sensitive assets and HYG: SPY, EEM, EFA, VNQ, and HYG have positive significant gamma estimates and better information criteria than symmetric Student-t GARCH.

The evidence is weaker for defensive and diversifying assets. IEF does not benefit from the asymmetric term, and DBC/TLT show statistical significance with limited information-criterion improvement. GLD has a small negative gamma, so the standard leverage interpretation does not apply cleanly.

This argues against selecting GJR automatically for the whole universe. It may fit some assets better in sample, but the final decision should be based on OOS forecast quality and robustness.


### Out-of-Sample Evaluation

The candidate GARCH specifications are evaluated on post-2018 forecasts against the same 21-day forward realized-volatility target. Parameters are estimated through the in-sample cutoff. These tables are validation diagnostics; they should not be described as a second round of strategy tuning.


In [66]:
garch_t_oos_vol, garch_t_oos_models = garch_oos_forecast(
    returns_data=returns_pct,
    model_type="GARCH",
    p=1,
    o=0,
    q=1,
    dist="t",
    is_end=IS_END,
    oos_start=OOS_START,
    horizon=HORIZON
)

In [67]:
gjr_t_oos_vol, gjr_t_oos_models = garch_oos_forecast(
    returns_data=returns_pct,
    model_type="GARCH",
    p=1,
    o=1,
    q=1,
    dist="t",
    is_end=IS_END,
    oos_start=OOS_START,
    horizon=HORIZON
)

In [30]:
garch_norm_oos_vol, garch_norm_oos_models = garch_oos_forecast(
    returns_data=returns_pct,
    model_type="GARCH",
    p=1,
    o=0,
    q=1,
    dist="normal",
    is_end=IS_END,
    oos_start=OOS_START,
    horizon=HORIZON
)

In [31]:
garch_norm_oos_eval = evaluate_vol_forecast(
    vol_forecast=garch_norm_oos_vol,
    realized_vol=realized_vol_oos
)

garch_t_oos_eval = evaluate_vol_forecast(
    vol_forecast=garch_t_oos_vol,
    realized_vol=realized_vol_oos
)

gjr_t_oos_eval = evaluate_vol_forecast(
    vol_forecast=gjr_t_oos_vol,
    realized_vol=realized_vol_oos
)

### Forecast Error Metrics


In [32]:
garch_oos_comparison = pd.DataFrame({
    "GARCH-N MAE": garch_norm_oos_eval["MAE"],
    "GARCH-t MAE": garch_t_oos_eval["MAE"],
    "GJR-t MAE": gjr_t_oos_eval["MAE"],

    "GARCH-N RMSE": garch_norm_oos_eval["RMSE"],
    "GARCH-t RMSE": garch_t_oos_eval["RMSE"],
    "GJR-t RMSE": gjr_t_oos_eval["RMSE"],

    "GARCH-N Corr": garch_norm_oos_eval["Correlation"],
    "GARCH-t Corr": garch_t_oos_eval["Correlation"],
    "GJR-t Corr": gjr_t_oos_eval["Correlation"],

    "GARCH-N QLIKE": garch_norm_oos_eval["QLIKE"],
    "GARCH-t QLIKE": garch_t_oos_eval["QLIKE"],
    "GJR-t QLIKE": gjr_t_oos_eval["QLIKE"]
}).round(4)

garch_oos_comparison.loc[risk_model_assets]

,GARCH-N MAE,GARCH-t MAE,GJR-t MAE,GARCH-N RMSE,GARCH-t RMSE,GJR-t RMSE,GARCH-N Corr,GARCH-t Corr,GJR-t Corr,GARCH-N QLIKE,GARCH-t QLIKE,GJR-t QLIKE
SPY,0.0565,0.0614,0.0596,0.0931,0.0998,0.0960,0.5436,0.5417,0.5844,0.4352,0.4473,0.4545
EFA,0.0531,0.0527,0.0520,0.0901,0.0899,0.0887,0.4856,0.4833,0.5131,0.3418,0.3455,0.3394
EEM,0.0587,0.0582,0.0583,0.0946,0.0939,0.0954,0.4439,0.4419,0.4562,0.2627,0.2628,0.2723
IEF,0.0149,0.0149,0.0150,0.0224,0.0225,0.0226,0.6136,0.6149,0.6103,0.1968,0.1973,0.1975
TLT,0.0329,0.0329,0.0338,0.0566,0.0573,0.0583,0.4769,0.4765,0.4645,0.2217,0.2226,0.2186
LQD,0.0231,0.0217,0.0213,0.0485,0.0501,0.0493,0.5155,0.4675,0.4780,0.5984,0.7838,0.8366
HYG,0.0277,0.0266,0.0250,0.0467,0.0462,0.0439,0.6845,0.6854,0.7195,0.5125,0.5067,0.4892
GLD,0.0439,0.0434,0.0431,0.0624,0.0627,0.0620,0.5303,0.5268,0.5397,0.2400,0.2435,0.2342
DBC,0.0466,0.0473,0.0486,0.0659,0.0669,0.0688,0.4427,0.4429,0.4337,0.2714,0.2775,0.2889
VNQ,0.0588,0.0590,0.0585,0.1077,0.1093,0.1098,0.5632,0.5586,0.5657,0.3977,0.4185,0.4382


The OOS results should be read asset by asset rather than as a single ranking. The preferred specification varies by metric, and the more flexible models do not dominate consistently across the universe.

This matters for portfolio construction: a model with slightly better in-sample fit is not automatically the best allocation input. The selection should prioritize stable forecasts, acceptable OOS errors, and interpretable behavior across asset classes.


### Out-of-Sample Calibration Regressions


In [33]:
garch_norm_oos_regression = vol_forecast_regression_hac(
    vol_forecast=garch_norm_oos_vol,
    realized_vol=realized_vol_oos,
    maxlags=HORIZON
)

garch_t_oos_regression = vol_forecast_regression_hac(
    vol_forecast=garch_t_oos_vol,
    realized_vol=realized_vol_oos,
    maxlags=HORIZON
)

gjr_t_oos_regression = vol_forecast_regression_hac(
    vol_forecast=gjr_t_oos_vol,
    realized_vol=realized_vol_oos,
    maxlags=HORIZON
)

In [34]:
garch_oos_regression_comparison = pd.DataFrame({
    "GARCH-N Beta": garch_norm_oos_regression["Beta"],
    "GARCH-t Beta": garch_t_oos_regression["Beta"],
    "GJR-t Beta": gjr_t_oos_regression["Beta"],

    "GARCH-N R2": garch_norm_oos_regression["R2"],
    "GARCH-t R2": garch_t_oos_regression["R2"],
    "GJR-t R2": gjr_t_oos_regression["R2"],

    "GARCH-N Joint p-value": garch_norm_oos_regression["Joint p-value"],
    "GARCH-t Joint p-value": garch_t_oos_regression["Joint p-value"],
    "GJR-t Joint p-value": gjr_t_oos_regression["Joint p-value"]
}).round(4)

garch_oos_regression_comparison.loc[risk_model_assets]

,GARCH-N Beta,GARCH-t Beta,GJR-t Beta,GARCH-N R2,GARCH-t R2,GJR-t R2,GARCH-N Joint p-value,GARCH-t Joint p-value,GJR-t Joint p-value
Asset,,,,,,,,,
SPY,0.6467,0.5595,0.5845,0.2955,0.2934,0.3415,0.0027,0.0000,0.0000
EFA,0.5099,0.5097,0.5169,0.2358,0.2335,0.2633,0.0000,0.0000,0.0000
EEM,0.5158,0.5230,0.4916,0.1970,0.1953,0.2082,0.0000,0.0000,0.0000
IEF,0.8350,0.8034,0.7978,0.3766,0.3781,0.3724,0.2038,0.1465,0.1476
TLT,0.7072,0.6661,0.6289,0.2275,0.2271,0.2158,0.1781,0.0742,0.0421
LQD,0.6731,0.6341,0.6709,0.2658,0.2185,0.2284,0.0265,0.0216,0.0345
HYG,0.7280,0.7226,0.7310,0.4686,0.4698,0.5176,0.0002,0.0014,0.0004
GLD,0.7415,0.7088,0.7317,0.2813,0.2775,0.2912,0.0073,0.0063,0.0146
DBC,0.5649,0.5407,0.4997,0.1959,0.1962,0.1881,0.0000,0.0000,0.0000


The OOS calibration regressions show whether forecast levels move with realized volatility after 2018. Positive beta estimates indicate that the forecasts retain timing information. Slopes below one imply compressed forecast variation relative to the realized target, so the models pick up regimes but understate the size of some volatility moves.

The joint calibration tests are stricter than the error metrics and should not be expected to pass uniformly. Rejections mean the forecasts are useful conditional-risk estimates, not statistically unbiased forecasts of the realized-volatility proxy.

Together with the error metrics, this supports a conservative selection: added flexibility is useful only where it improves OOS behavior without creating convergence or stability problems.


## 4. Final Volatility Model Selection

The final volatility input uses Student-t GARCH(1,1) for the risk assets and EWMA/RiskMetrics for SHY.

The rationale is deliberately practical. ARCH-LM tests support conditional-volatility modelling, Student-t innovations improve in-sample fit for the risk assets, and a common symmetric GARCH specification is easier to defend than selecting a different model for every ETF. GJR-GARCH shows asymmetry for some equity- and credit-sensitive assets, but the extra parameter is not used in the final portfolio input because the benefit is uneven and the specification is less stable.

SHY is handled separately because its GARCH estimates show convergence and boundary problems. Using EWMA for the cash proxy avoids forcing an unstable parametric model onto a very low-volatility series.


In [35]:
# Full-period selected volatility model:
#   - IS: fitted conditional volatility from the Student-t GARCH(1,1) models
#   - OOS: fixed-parameter Student-t GARCH(1,1) forecasts

final_vol_model_is = pd.DataFrame({
    asset: model.conditional_volatility / 100 * np.sqrt(TRADING_DAYS)
    for asset, model in garch_t_models.items()
}).loc[:IS_END]

final_vol_model = pd.concat([
    final_vol_model_is,
    garch_t_oos_vol.loc[OOS_START:]
])

final_vol_model = (
    final_vol_model
    .sort_index()
    .loc[~final_vol_model.index.duplicated(keep="last")]
    .dropna(how="all")
)

# SHY uses EWMA instead of GARCH over the whole available period.
if CASH_PROXY in final_vol_model.columns:
    common_idx = final_vol_model.index.intersection(ewma_vol_lagged.index)
    final_vol_model.loc[common_idx, CASH_PROXY] = ewma_vol_lagged.loc[common_idx, CASH_PROXY]

forecast_sanity_table(final_vol_model)

,Min,Median,Mean,Max
SPY,0.0575,0.1470,0.1755,1.0699
EFA,0.0648,0.1620,0.1935,1.1580
EEM,0.1080,0.2027,0.2427,1.5715
IEF,0.0360,0.0621,0.0670,0.1671
TLT,0.0726,0.1371,0.1453,0.4892
LQD,0.0381,0.0588,0.0696,0.5675
HYG,0.0273,0.0640,0.0863,0.9608
GLD,0.1007,0.1563,0.1727,0.4906
DBC,0.0756,0.1693,0.1827,0.5180
VNQ,0.0799,0.1762,0.2353,1.6171


The final volatility input is now saved for the full available period. Before the OOS cutoff, it uses the fitted in-sample Student-t GARCH(1,1) conditional volatility. From the OOS start onward, it uses fixed-parameter Student-t GARCH(1,1) forecasts estimated through the in-sample cutoff.

SHY is handled separately because the Student-t and GJR estimates generated convergence and boundary issues. Using EWMA for SHY avoids forcing a parametric model onto a cash proxy where the GARCH likelihood is unstable.

The full-period file is used by the portfolio notebook when adding an in-sample parameter-selection step. The OOS-only files are still saved separately for strict post-2018 evaluation.

In [36]:
final_vol_forecast = final_vol_model.loc[OOS_START:].dropna(how="all")
final_vol_model.tail()

,SPY,EFA,EEM,IEF,TLT,LQD,HYG,GLD,DBC,VNQ,SHY
Date,,,,,,,,,,,
2026-06-03,0.114231,0.158830,0.269169,0.053862,0.105989,0.055849,0.052274,0.225034,0.220899,0.150971,0.012621
2026-06-04,0.111217,0.156697,0.264337,0.053048,0.104686,0.055000,0.051279,0.221862,0.220750,0.168419,0.012382
2026-06-05,0.188149,0.201471,0.396441,0.054960,0.104711,0.058686,0.059676,0.246388,0.228059,0.164694,0.012338
2026-06-08,0.177567,0.194129,0.386696,0.054141,0.104750,0.057515,0.057299,0.241348,0.224612,0.173647,0.014423
2026-06-09,0.193250,0.190395,0.374071,0.053547,0.103923,0.056322,0.054943,0.242003,0.223205,0.191688,0.014112


In [37]:
# Month-end volatility inputs for the portfolio-allocation notebook.
# The full-period version supports in-sample model selection in Notebook 4.

final_vol_model_monthly = final_vol_model.resample("ME").last().dropna(how="all")
final_vol_forecast_monthly = final_vol_forecast.resample("ME").last().dropna(how="all")

final_vol_model_monthly.head()

,SPY,EFA,EEM,IEF,TLT,LQD,HYG,GLD,DBC,VNQ,SHY
Date,,,,,,,,,,,
2007-04-30,0.089424,0.115041,0.160036,0.036929,0.079181,0.040311,0.044633,0.154822,0.150048,0.135433,0.008698
2007-05-31,0.099141,0.117801,0.202457,0.037833,0.079501,0.040488,0.039707,0.144801,0.158044,0.270414,0.009874
2007-06-30,0.151065,0.142891,0.206229,0.050491,0.105518,0.053867,0.070907,0.143114,0.142580,0.254411,0.012441
2007-07-31,0.222495,0.228252,0.362240,0.056881,0.109223,0.052432,0.154367,0.136516,0.137165,0.282926,0.017486
2007-08-31,0.219488,0.241615,0.407503,0.057810,0.098277,0.066078,0.088678,0.139303,0.127943,0.360258,0.022273


In [38]:
final_vol_model.to_csv(OUTPUT_DIR / "final_selected_vol_model_daily.csv")
final_vol_model_monthly.to_csv(OUTPUT_DIR / "final_selected_vol_model_monthly.csv")

final_vol_forecast.to_csv(OUTPUT_DIR / "final_selected_vol_forecast_daily_oos.csv")
final_vol_forecast_monthly.to_csv(OUTPUT_DIR / "final_selected_vol_forecast_monthly_oos.csv")

print("Saved full-period and OOS-only volatility inputs to:", OUTPUT_DIR.resolve())

Saved full-period and OOS-only volatility inputs to: C:\Users\mmodr\Desktop\project-trend-invvol-strategy\results\volatility_forecast
